In [1]:
import json
import warnings
import numpy as np
import pandas as pd
import optuna
import torch

warnings.filterwarnings("ignore")

# SDV Imports (Thư viện chuẩn mực cho Tabular GenAI)
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer
from sdv.sampling import Condition
from sdv.evaluation.single_table import evaluate_quality

# =========================
# 1) Config & Hyperparameters
# =========================
SEED = 121
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_PATH = r"C:\Users\NFU_Power_Lab\Desktop\DAT_ResearchArea\BestAwardAI2026\random-split\wpt_train_random.csv"
OUT_AUG_PATH = "wpt_train_random_aug_ctgan.csv"          
OUT_SYN_ONLY_PATH = "wpt_train_random_syn_only_ctgan.csv" 
OPTUNA_PARAMS_PATH = "best_ctgan_params.json"           

# Balancing Policy (Controlled Generation)
MAX_SYN_MULTIPLIER = 2.0 
TARGET_MINIMUM = 50      
SYN_PER_TYPE_CAP = 300    

# Optuna Config
OPTUNA_TRIALS = 30
FINAL_EPOCHS = 500

# =========================
# 2) Load & Preprocess Data
# =========================
df = pd.read_csv(TRAIN_PATH)
expected = {"type", "load", "efficiency"}
if expected - set(df.columns):
    raise ValueError(f"Missing columns: {expected - set(df.columns)}")

df["type"] = df["type"].astype(str)
df["load"] = pd.to_numeric(df["load"], errors="coerce")
df["efficiency"] = pd.to_numeric(df["efficiency"], errors="coerce")
df = df.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)

print("Train shape:", df.shape)

# Chia tập validation nhỏ để Optuna đánh giá nhanh
val_ratio = 0.2
val_n = int(len(df) * val_ratio)
df_train = df.iloc[val_n:].reset_index(drop=True)
df_val = df.iloc[:val_n].reset_index(drop=True)

# =========================
# 3) Build SDV Metadata
# =========================
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)
metadata.update_column(column_name='type', sdtype='categorical')
metadata.update_column(column_name='load', sdtype='numerical')
metadata.update_column(column_name='efficiency', sdtype='numerical')

# =========================
# 4) Setup Optuna Objective Function cho CTGAN
# =========================
def objective(trial):
    # Không gian siêu tham số của CTGAN
    embedding_dim = trial.suggest_categorical("embedding_dim", [64, 128])
    batch_size = trial.suggest_categorical("batch_size", [50, 100, 200]) # CTGAN thích batch_size lớn
    epochs = trial.suggest_int("epochs", 100, 300, step=50) 
    
    # Cấu trúc mạng Generator
    g_dim_choice = trial.suggest_categorical("g_dims", [1, 2])
    generator_dim = (256, 256) if g_dim_choice == 1 else (256, 256, 256)

    # Cấu trúc mạng Discriminator
    d_dim_choice = trial.suggest_categorical("d_dims", [1, 2])
    discriminator_dim = (256, 256) if d_dim_choice == 1 else (256, 256, 256)

    # Khởi tạo mô hình CTGAN
    synthesizer = CTGANSynthesizer(
        metadata,
        embedding_dim=embedding_dim,
        generator_dim=generator_dim,
        discriminator_dim=discriminator_dim,
        batch_size=batch_size,
        epochs=epochs,
        cuda=True if torch.cuda.is_available() else False,
        verbose=False
    )
    
    # Huấn luyện
    synthesizer.fit(df_train)
    
    try:
        syn_val = synthesizer.sample(num_rows=len(df_val))
        quality_report = evaluate_quality(
            real_data=df_val,
            synthetic_data=syn_val,
            metadata=metadata,
            verbose=False
        )
        score = quality_report.get_score()
    except Exception as e:
        score = 0.0

    return score

# =========================
# 5) Run Optuna Optimization
# =========================
print(f"\n=== Bắt đầu tìm kiếm Optuna ({OPTUNA_TRIALS} Trials) ===")
study = optuna.create_study(direction="maximize") 
study.optimize(objective, n_trials=OPTUNA_TRIALS)

print("\n=== Optuna Hoàn Thành ===")
best_params = study.best_params
print(f"Điểm Quality Score cao nhất: {study.best_value:.4f}")
print(f"Các tham số tốt nhất: {best_params}")

with open(OPTUNA_PARAMS_PATH, 'w') as f:
    json.dump(best_params, f, indent=4)
print(f"Đã lưu siêu tham số vàng tại: {OPTUNA_PARAMS_PATH}")

# =========================
# 6) Train Final CTGAN Model
# =========================
print(f"\n=== Bắt đầu huấn luyện mô hình CTGAN cuối cùng ({FINAL_EPOCHS} Epochs) ===")
final_g_dims = (256, 256) if best_params["g_dims"] == 1 else (256, 256, 256)
final_d_dims = (256, 256) if best_params["d_dims"] == 1 else (256, 256, 256)

final_synthesizer = CTGANSynthesizer(
    metadata,
    embedding_dim=best_params["embedding_dim"],
    generator_dim=final_g_dims,
    discriminator_dim=final_d_dims,
    batch_size=best_params["batch_size"],
    epochs=FINAL_EPOCHS,
    cuda=True if torch.cuda.is_available() else False,
    verbose=True
)

final_synthesizer.fit(df)
print("Huấn luyện thành công!")

# =========================
# 7) Generate Synthetic Data (Controlled Balancing)
# =========================
counts = df["type"].value_counts().sort_index()
print("\nCounts by type (before):\n", counts)

syn_parts = []
eff_min, eff_max = df["efficiency"].min(), df["efficiency"].max()

for t in sorted(df["type"].unique()):
    n_real = int((df["type"] == t).sum())

    # Bỏ qua Class lớn
    if t == 'K' or n_real > 300:
        continue

    desired_target = max(TARGET_MINIMUM, int(counts.median()))
    need = desired_target - n_real
    max_allowed_syn = int(n_real * MAX_SYN_MULTIPLIER)
    need = min(need, max_allowed_syn)
    need = min(need, SYN_PER_TYPE_CAP)
 
    if need <= 0:
        continue

    print(f"Đang sinh {need} mẫu giả cho class: {t}...")
    condition = Condition(num_rows=need, column_values={'type': t})
    
    syn_df = final_synthesizer.sample_from_conditions(conditions=[condition])
    
    # Clip lại hiệu suất cho đúng giới hạn vật lý
    syn_df["efficiency"] = np.clip(syn_df["efficiency"], eff_min, eff_max)
    syn_df["is_synthetic"] = 1
    
    syn_parts.append(syn_df)

# =========================
# 8) Merge & Save
# =========================
if syn_parts:
    syn_df_all = pd.concat(syn_parts, ignore_index=True)
else:
    syn_df_all = pd.DataFrame(columns=["type","load","efficiency","is_synthetic"])

real_df = df.copy()
real_df["is_synthetic"] = 0
aug_df = pd.concat([real_df, syn_df_all], ignore_index=True)

before = df["type"].value_counts().sort_index()
after = aug_df["type"].value_counts().sort_index()
report = pd.DataFrame({"before": before, "after": after}).fillna(0).astype(int)

report.to_csv("ctgan_type_counts_before_after.csv", encoding="utf-8-sig")
aug_df.to_csv(OUT_AUG_PATH, index=False, encoding="utf-8-sig")

if not syn_df_all.empty:
    syn_df_all.to_csv(OUT_SYN_ONLY_PATH, index=False, encoding="utf-8-sig")

print("\nGenerated synthetic rows:", len(syn_df_all))
print("Saved augmented train:", OUT_AUG_PATH)
if not syn_df_all.empty:
    print("Saved synthetic-only data:", OUT_SYN_ONLY_PATH)
print("\nCounts by type (after):\n", after)

# =========================
# 9) Xuất File Thống kê Phân phối
# =========================
stats_before = df.groupby("type")["efficiency"].agg(["mean","std","count"]).rename(columns={"count":"n_before"})
stats_after  = aug_df.groupby("type")["efficiency"].agg(["mean","std","count"]).rename(columns={"count":"n_after"})
stats_merge = stats_before.join(stats_after, lsuffix="_real", rsuffix="_aug")
stats_merge.to_csv("ctgan_type_stats_before_after.csv", encoding="utf-8-sig")
print("Saved: ctgan_type_stats_before_after.csv")

[I 2026-03-10 23:34:28,974] A new study created in memory with name: no-name-2270203a-9bac-49ba-94c3-35b78289c229


Train shape: (1310, 3)

=== Bắt đầu tìm kiếm Optuna (30 Trials) ===


[I 2026-03-10 23:36:39,488] Trial 0 finished with value: 0.7629112352402645 and parameters: {'embedding_dim': 64, 'batch_size': 50, 'epochs': 200, 'g_dims': 2, 'd_dims': 2}. Best is trial 0 with value: 0.7629112352402645.
[I 2026-03-10 23:38:11,261] Trial 1 finished with value: 0.7449922683863562 and parameters: {'embedding_dim': 128, 'batch_size': 50, 'epochs': 150, 'g_dims': 2, 'd_dims': 2}. Best is trial 0 with value: 0.7629112352402645.
[I 2026-03-10 23:40:53,444] Trial 2 finished with value: 0.7502551931673005 and parameters: {'embedding_dim': 64, 'batch_size': 50, 'epochs': 300, 'g_dims': 1, 'd_dims': 2}. Best is trial 0 with value: 0.7629112352402645.
[I 2026-03-10 23:41:31,961] Trial 3 finished with value: 0.7121062433244274 and parameters: {'embedding_dim': 64, 'batch_size': 200, 'epochs': 300, 'g_dims': 1, 'd_dims': 1}. Best is trial 0 with value: 0.7629112352402645.
[I 2026-03-10 23:42:12,384] Trial 4 finished with value: 0.6704955748342074 and parameters: {'embedding_dim': 


=== Optuna Hoàn Thành ===
Điểm Quality Score cao nhất: 0.7686
Các tham số tốt nhất: {'embedding_dim': 64, 'batch_size': 100, 'epochs': 300, 'g_dims': 2, 'd_dims': 2}
Đã lưu siêu tham số vàng tại: best_ctgan_params.json

=== Bắt đầu huấn luyện mô hình CTGAN cuối cùng (500 Epochs) ===


Gen. (-01.27) | Discrim. (+00.07): 100%|██████████| 500/500 [03:06<00:00,  2.68it/s]


...

Đang sinh 52 mẫu giả cho class: F...


Sampling conditions: 100%|██████████| 52/52 [00:00<00:00, 658.00it/s]


Đang sinh 46 mẫu giả cho class: I...


Sampling conditions: 100%|██████████| 46/46 [00:00<00:00, 289.20it/s]



Generated synthetic rows: 192
Saved augmented train: wpt_train_random_aug_ctgan.csv
Saved synthetic-only data: wpt_train_random_syn_only_ctgan.csv

Counts by type (after):
 type
A     66
B     91
C     72
D     95
E     91
F     78
G     95
H    279
I     69
J    185
K    381
Name: count, dtype: int64
Saved: ctgan_type_stats_before_after.csv
